# M2

In [ ]:
import os
import sys
from pathlib import Path

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the code folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from simulation.run_simulation import SimulationConfig, run_simulation
from utils import ratio_table
from IPython.display import HTML


In [ ]:
n = 500  # Number of labeled observations (X and y).
N = 5000  # Number of unlabeled observations (X only).
reps = 100  # Number of repeated experiments for each setting.
dimensions = range(2, 16)  # Numbers of input columns to test, from 2 to 15.
seed = 1  # Random seed for the first repetition.
noise_sigma = 1.0  # Standard deviation of random noise added to the response.
# Split the training data into cv_folds groups to compare tuning candidates.
# Fit on all but one group and measure prediction error on the remaining group.
cv_folds = 10
jobs = 12  # Maximum number of repetitions run at the same time.
estimators = ["ata_sbf", "ata_csbf", "krr_m"]  # Methods to fit to each repeated sample.

experiment_m2 = SimulationConfig(
    model="M2",
    estimators=estimators,
    dimensions=dimensions,
    n=n,
    N=N,
    reps=reps,
    seed=seed,
    noise_sigma=noise_sigma,
    cv_folds=cv_folds,
    jobs=jobs,
    run_name=f"M2_n{n}_N{N}",  # Output folder name for this run.
)

summary_m2, records_m2 = run_simulation(experiment_m2)

In [ ]:
table_m2 = ratio_table(summary_m2, "M2")
shown_m2 = table_m2.map(lambda value: f"{value:.3f}")
shown_m2 = shown_m2.mask(
    table_m2.eq(table_m2.max()), "<b>" + shown_m2 + "</b>"
)
HTML(shown_m2.to_html(escape=False))